# IT3091 — Group 30 — Member 1: Data Understanding & EDA

**Scenario:** Code 0, Retail & E-commerce
**Dataset:** UCI Online Retail (ID 352) — https://archive.ics.uci.edu/dataset/352/online-retail
**Lens:** Customer Segmentation Based on Purchasing Behaviour

**How to use this notebook**

Run every cell from top to bottom. Each section ends with a cell printing
**Observation / Meaning / Impact**. Copy those into your EDA insight log, but
rewrite them in your own words after you look at the chart — do not paste them blindly.

> Before you start: download `Online Retail.xlsx` from the UCI link and upload it.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib, os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f"{FIGDIR}/{name}.png", bbox_inches="tight")
    plt.show()

def insight(obs, meaning, impact):
    print("OBSERVATION:", obs)
    print("MEANING    :", meaning)
    print("IMPACT     :", impact)

### Load the data and record its fingerprint

The fingerprint (a SHA-256 hash) proves everyone used the identical file.
The reproducibility criterion asks for this — it is 10 marks and takes 5 lines.

**In Colab:** run the cell below, then click *Choose Files* and pick `Online Retail.xlsx`.
**On your own machine:** put the file beside this notebook and skip the upload.

In [ ]:
PATH = "Online Retail.xlsx"

if not os.path.exists(PATH):
    try:
        from google.colab import files
        up = files.upload()
        PATH = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError("Put 'Online Retail.xlsx' next to this notebook.")

with open(PATH, "rb") as f:
    FINGERPRINT = hashlib.sha256(f.read()).hexdigest()

df = pd.read_excel(PATH)          # takes 1-2 minutes, it is a 23 MB Excel file
df.to_pickle("online_retail_raw.pkl")   # faster to reload next time

print("File        :", PATH)
print("SHA-256     :", FINGERPRINT)
print("Rows, cols  :", df.shape)
print("Date range  :", df.InvoiceDate.min(), "to", df.InvoiceDate.max())

> **For your report:** record the file name, the SHA-256 above, and the row count.
> Anyone re-running this should get the same hash.

## 1. What does one row mean?

This is the first thing a marker looks for. A row is **one product line on one invoice** —
not one customer and not one order. One invoice spans several rows.

In [ ]:
display(df.head(8))
print("\nOne invoice spans several rows. Example:")
example = df.InvoiceNo.value_counts().index[0]
display(df[df.InvoiceNo == example].head(6))
print(f"\nInvoice {example} has {(df.InvoiceNo == example).sum()} rows (product lines).")

In [ ]:
print("Rows (product lines):", f"{len(df):,}")
print("Unique invoices     :", f"{df.InvoiceNo.nunique():,}")
print("Unique customers    :", f"{df.CustomerID.nunique():,}")
print("Unique products     :", f"{df.StockCode.nunique():,}")
print("Countries           :", df.Country.nunique())

## 2. Data dictionary (raw fields)

In [ ]:
dd = pd.DataFrame({
    "Column": df.columns,
    "Type": [str(t) for t in df.dtypes],
    "Missing": [df[c].isna().sum() for c in df.columns],
    "Missing %": [round(df[c].isna().mean()*100, 2) for c in df.columns],
    "Distinct": [df[c].nunique() for c in df.columns],
    "Example": [df[c].dropna().iloc[0] for c in df.columns],
})
dd["Meaning"] = [
    "Invoice number. Starting with 'C' means the order was cancelled",
    "Product code",
    "Product name",
    "How many items were bought (negative = returned)",
    "Date and time of the sale",
    "Price of one item, in British pounds",
    "Customer number",
    "Country where the customer lives",
]
display(dd)
dd.to_csv("data_dictionary_raw.csv", index=False)

## Q1. How many rows have no customer number?

In [ ]:
miss = df.isna().sum()
miss = miss[miss > 0]

ax = miss.plot(kind="bar", color="#4C72B0")
ax.set_title("Missing values by column")
ax.set_ylabel("Number of rows")
for i, v in enumerate(miss):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
save("q1_missing_values")

no_cust = df.CustomerID.isna().sum()
print(f"Rows with no CustomerID: {no_cust:,}  ({no_cust/len(df)*100:.1f}%)")
print(f"Rows usable for customer work: {len(df)-no_cust:,}")

insight(
 f"{no_cust:,} rows ({no_cust/len(df)*100:.1f}%) have no CustomerID.",
 "These are likely guest or till sales that cannot be traced to a person.",
 "Member 2 must drop them for customer-level work. Member 1 records this as a limitation: our segments describe identified customers only.")

## Q2. Is spending per customer even or lopsided?

This drives the single most important preprocessing decision in the project.

In [ ]:
df["LineTotal"] = df.Quantity * df.UnitPrice
spend = df.dropna(subset=["CustomerID"]).groupby("CustomerID").LineTotal.sum()
spend_pos = spend[spend > 0]

fig, ax = plt.subplots(1, 2)
ax[0].hist(spend_pos, bins=80, color="#4C72B0")
ax[0].set_title("Total spend per customer (raw)")
ax[0].set_xlabel("Pounds")
ax[1].hist(np.log10(spend_pos), bins=60, color="#55A868")
ax[1].set_title("Same data, log10 scale")
ax[1].set_xlabel("log10(pounds)")
save("q2_spend_skew")

print(spend_pos.describe())
top1 = spend_pos.nlargest(max(1, int(len(spend_pos)*0.01))).sum()
print(f"\nSkewness: {spend_pos.skew():.1f}  (0 = symmetric)")
print(f"Top 1% of customers account for {top1/spend_pos.sum()*100:.1f}% of revenue.")

insight(
 f"Spend is extremely right-skewed (skewness {spend_pos.skew():.1f}); the top 1% of customers bring in {top1/spend_pos.sum()*100:.1f}% of revenue.",
 "A small number of wholesalers spend far more than ordinary customers.",
 "Member 2 should log-transform Monetary before scaling. Member 3 must note that K-Means uses distances, so untransformed spend would let a few wholesalers dominate every cluster.")

## Q3. How many customers bought only once?

In [ ]:
orders = df.dropna(subset=["CustomerID"]).groupby("CustomerID").InvoiceNo.nunique()

ax = orders.clip(upper=15).value_counts().sort_index().plot(kind="bar", color="#4C72B0")
ax.set_title("Number of orders per customer (15+ grouped)")
ax.set_xlabel("Orders placed")
ax.set_ylabel("Customers")
save("q3_orders_per_customer")

once = (orders == 1).sum()
print(f"Customers with exactly 1 order: {once:,} ({once/len(orders)*100:.1f}%)")
print(f"Median orders per customer    : {orders.median():.0f}")
print(f"Most orders by one customer   : {orders.max()}")

insight(
 f"{once:,} customers ({once/len(orders)*100:.1f}%) placed only one order.",
 "A large one-time group exists alongside a smaller repeat-buying group.",
 "Member 3 should expect at least one cluster of one-time low-value customers. Member 4 should check this group really does stay away after the cut-off date.")

## Q4. When did customers last buy? (Recency)

In [ ]:
CUTOFF = pd.Timestamp("2011-09-09")
print("Cut-off date for building segments:", CUTOFF.date())
print("Data continues to:", df.InvoiceDate.max())
print("Days available after cut-off:", (df.InvoiceDate.max() - CUTOFF).days)

last = df.dropna(subset=["CustomerID"]).groupby("CustomerID").InvoiceDate.max()
recency = (df.InvoiceDate.max() - last).dt.days

ax = recency.plot(kind="hist", bins=60, color="#4C72B0")
ax.set_title("Days since last purchase (full data)")
ax.set_xlabel("Days")
save("q4_recency")

print(recency.describe())

insight(
 f"Recency ranges from {recency.min()} to {recency.max()} days; the median is {recency.median():.0f} days.",
 "Many customers have been quiet for a long time while others bought very recently.",
 "Recency separates customers well, so it is a strong clustering feature. Member 3 keeps it. Member 4 uses the 90 days after 9 Sep 2011 to test whether 'lost' customers really stayed away.")

## Q5. How do sales change over the year?

In [ ]:
monthly = df.set_index("InvoiceDate").LineTotal.resample("ME").sum()

ax = monthly.plot(marker="o", color="#4C72B0")
ax.set_title("Total revenue by month")
ax.set_ylabel("Pounds")
save("q5_monthly_revenue")

print(monthly)
print("\nNote: Dec 2010 and Dec 2011 are both partial months.")

insight(
 "Revenue climbs steeply to a peak in November 2011, then drops in the partial December.",
 "The business is highly seasonal around Christmas.",
 "Member 4 must remember the 90-day test window (Sep-Dec) contains the Christmas peak, so return rates there are higher than a normal quarter. This belongs in the limitations section.")

## Q6. Where do customers come from?

In [ ]:
top = df.Country.value_counts().head(10)

ax = top.plot(kind="barh", color="#4C72B0")
ax.set_title("Top 10 countries by number of rows")
ax.invert_yaxis()
save("q6_countries")

uk = (df.Country == "United Kingdom").mean() * 100
print(f"United Kingdom share of rows: {uk:.1f}%")
print(f"Distinct countries: {df.Country.nunique()}")
print("\nOdd values:", [x for x in df.Country.unique() if x in ("Unspecified","European Community","EIRE")])

insight(
 f"{uk:.1f}% of rows are from the United Kingdom, and some values are vague ('Unspecified', 'European Community').",
 "Country is badly imbalanced and partly unreliable.",
 "Member 2 should collapse Country to UK / non-UK rather than one-hot encoding 38 values, which would add noise to distance-based clustering.")

## Q7. How common are cancellations?

In [ ]:
df["IsCancel"] = df.InvoiceNo.astype(str).str.startswith("C")
canc_rows = df.IsCancel.sum()
canc_inv = df.loc[df.IsCancel, "InvoiceNo"].nunique()

print(f"Cancelled rows    : {canc_rows:,} ({canc_rows/len(df)*100:.2f}%)")
print(f"Cancelled invoices: {canc_inv:,}")
print(f"All cancelled rows have negative Quantity: {(df.loc[df.IsCancel,'Quantity'] < 0).all()}")
display(df[df.IsCancel].head(5))

cust = df.dropna(subset=["CustomerID"])
rate = cust.groupby("CustomerID").IsCancel.mean()
print(f"\nCustomers who ever cancelled: {(rate > 0).sum():,} of {len(rate):,}")

insight(
 f"{canc_rows:,} rows ({canc_rows/len(df)*100:.2f}%) are cancellations, all with negative quantities, and {(rate>0).sum():,} customers have cancelled at least once.",
 "Returns are a real and measurable part of buying behaviour, not just dirty data.",
 "Member 2 must exclude them from spending totals so returns are not counted as sales. Member 3 can optionally test a cancellation-rate feature as an extra behaviour measure.")

## Q8. Are there impossible values?

In [ ]:
print(df[["Quantity","UnitPrice"]].describe())

print(f"\nRows with Quantity <= 0 : {(df.Quantity <= 0).sum():,}")
print(f"Rows with UnitPrice <= 0: {(df.UnitPrice <= 0).sum():,}")
print(f"Rows with UnitPrice < 0 : {(df.UnitPrice < 0).sum():,}")

print("\nMost extreme rows:")
display(df.reindex(df.Quantity.abs().nlargest(4).index))

insight(
 f"Quantity runs from {df.Quantity.min():,} to {df.Quantity.max():,} and UnitPrice from {df.UnitPrice.min():,.2f} to {df.UnitPrice.max():,.2f}; the extremes are a huge order and its matching cancellation.",
 "Some rows are accounting corrections rather than genuine sales.",
 "Member 2 removes non-positive prices and quantities with a documented rule. Member 1 keeps the evidence that these extremes are paired order/cancellation rows, not data-entry typos.")

## Q9. Which stock codes are not products?

In [ ]:
codes = df.StockCode.astype(str)
odd = codes[~codes.str.match(r"^\d{5}")].value_counts().head(15)
display(odd)

for code_ in odd.index[:6]:
    desc = df.loc[codes == code_, "Description"].dropna().unique()[:2]
    print(f"{code_:<15} {desc}")

insight(
 f"{odd.sum():,} rows use non-product codes such as {', '.join(odd.index[:4])} for postage, discounts, manual adjustments and bank charges.",
 "Not every row is a product sale.",
 "Member 2 filters these codes out of product-level analysis. Member 1 notes that postage still affects total spend, so the group must decide whether it counts as Monetary value — this becomes a decision log entry.")

## Q10. Are products cheap or expensive?

In [ ]:
prices = df.loc[(df.UnitPrice > 0) & (df.UnitPrice < 50), "UnitPrice"]

ax = prices.plot(kind="hist", bins=80, color="#4C72B0")
ax.set_title("Unit price distribution (under \u00a350)")
ax.set_xlabel("Pounds")
save("q10_unit_price")

print(df.loc[df.UnitPrice > 0, "UnitPrice"].describe())
print(f"\nItems under \u00a35: {(df.UnitPrice.between(0.01,5)).mean()*100:.1f}% of rows")

insight(
 f"Most items are cheap gifts; the median unit price is about \u00a3{df.loc[df.UnitPrice>0,'UnitPrice'].median():.2f}.",
 "High customer spend comes from buying in bulk, not from buying expensive items.",
 "This supports the wholesaler explanation for the skew in Q2, and helps Member 4 name the clusters ('bulk buyers' rather than 'luxury buyers').")

## Q11. Do the three RFM measures overlap?

If two measures say the same thing, the clustering is effectively using fewer dimensions
than you claim. Member 3 needs to know this.

In [ ]:
clean = df[(~df.IsCancel) & df.CustomerID.notna() & (df.Quantity > 0) & (df.UnitPrice > 0)].copy()
snapshot = clean.InvoiceDate.max() + pd.Timedelta(days=1)

rfm = clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda s: (snapshot - s.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("LineTotal", "sum"),
)
print("Preview RFM table (full data — Member 3 will rebuild this with the cut-off):")
display(rfm.describe())

corr = rfm.corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation between R, F and M (Spearman)")
save("q11_rfm_correlation")

print(corr)

insight(
 f"Frequency and Monetary are strongly correlated (Spearman {corr.loc['Frequency','Monetary']:.2f}), while Recency is negatively related to both.",
 "F and M partly measure the same underlying loyalty, so the three measures are not independent.",
 "Member 3 should scale all three so no single measure dominates, and should mention this overlap when interpreting clusters. Adding more spend-related features would worsen the redundancy.")

## Q12. Are there exact duplicate rows?

In [ ]:
dups = df.duplicated().sum()
print(f"Exact duplicate rows: {dups:,} ({dups/len(df)*100:.2f}%)")
display(df[df.duplicated(keep=False)].sort_values(list(df.columns[:6])).head(6))

insight(
 f"{dups:,} rows are exact duplicates of another row.",
 "The same product line was recorded twice, so quantities and totals are slightly inflated.",
 "Member 2 applies drop_duplicates() before building any totals, otherwise Frequency and Monetary are overstated for affected customers.")

## Summary table for your EDA insight log

Run this, then copy the table into your log document and rewrite the wording yourself.

In [ ]:
log = pd.DataFrame([
 ["Q1","Missing CustomerID","25% of rows have no customer","Guest/till sales, untraceable","M2 drops them; M1 records as limitation"],
 ["Q2","Spend skew","Top 1% dominate revenue","Wholesalers among ordinary customers","M2 log-transform; M3 distance-based methods affected"],
 ["Q3","One-time buyers","Large single-order group","Two populations exist","M3 expects a one-time cluster; M4 verifies it"],
 ["Q4","Recency spread","Wide range of last-purchase dates","Good separating power","M3 keeps Recency; M4 tests after cut-off"],
 ["Q5","Seasonality","November peak","Christmas-driven business","M4 notes test window includes the peak"],
 ["Q6","Country imbalance","91% UK","Country adds little information","M2 collapses to UK / non-UK"],
 ["Q7","Cancellations","~2% of rows, negative quantity","Returns are real behaviour","M2 excludes from totals; M3 may add a return-rate feature"],
 ["Q8","Impossible values","Negative prices, extreme quantities","Accounting corrections","M2 removes with a documented rule"],
 ["Q9","Non-product codes","POST, D, M, BANK CHARGES","Not all rows are sales","M2 filters them; group decides on postage"],
 ["Q10","Low unit prices","Mostly cheap gifts","Spend comes from bulk, not luxury","M4 names clusters accordingly"],
 ["Q11","R, F, M overlap","F and M strongly correlated","Measures are not independent","M3 scales all three; avoid more spend features"],
 ["Q12","Duplicate rows","Exact duplicates present","Totals slightly inflated","M2 drops duplicates first"],
], columns=["Ref","Topic","Observation","Meaning","Impact"])

display(log)
log.to_csv("eda_insight_log.csv", index=False)
print("\nSaved: eda_insight_log.csv")
print("Figures saved in:", FIGDIR)
print("Dataset fingerprint:", FINGERPRINT)

## Handover to the rest of the group

Give Member 2 the insight log and the figures folder. Their preprocessing decisions
should each cite a Q-number from this notebook as the evidence.

**Before you hand over:** restart the kernel and run everything top to bottom once,
so the outputs in your report match the notebook exactly.